In [1]:
import os
import shutil
import numpy as np
from config import Config
from env import MySumoEnv

_NEEDED_WORKER_FILES = ("detectors.add.xml", "network.net.xml", "run.sumocfg")

def prepare_worker_dir(worker_idx=0):
    if hasattr(Config, "ensure_dirs"):
        Config.ensure_dirs()
    else:
        os.makedirs(Config.WORKERS_ROOT, exist_ok=True)
    src = Config.BASE_WORK_DIR
    dst = Config.worker_dir(worker_idx)
    os.makedirs(dst, exist_ok=True)
    for name in _NEEDED_WORKER_FILES:
        src_path = os.path.join(src, name)
        dst_path = os.path.join(dst, name)
        if not os.path.exists(src_path):
            raise FileNotFoundError(f"No utils file: {src_path}")
        shutil.copy2(src_path, dst_path)
    os.makedirs(os.path.join(dst, "dump"), exist_ok=True)
    return dst

def build_env(worker_idx=0, total_step=None, seed=42):
    rl_dir = prepare_worker_dir(worker_idx)
    answer_path = Config.answer_path_for(worker_idx)
    sumocfg_path = os.path.join(rl_dir, "run.sumocfg")

    if not os.path.exists(rl_dir):
        raise FileNotFoundError(f"No directory: {rl_dir}")
    if not os.path.exists(sumocfg_path):
        raise FileNotFoundError(f"No run.sumocfg: {sumocfg_path}")
    if not os.path.exists(answer_path):
        raise FileNotFoundError(f"No answer.csv: {answer_path}")

    if total_step is None:
        total_step = Config.TOTAL_STEP

    env = MySumoEnv(
        rl_dir=rl_dir,
        sumo_binary=Config.SUMO_BINARY,
        origin_list=Config.ORIGIN_LIST,
        destination_list=Config.DESTINATION_LIST,
        input_interval=Config.INPUT_INTERVAL,
        detector_interval=Config.DETECTOR_INTERVAL,
        num_OD=Config.NUM_OD,
        state_dim=1,
        answer_dir=answer_path,
        total_step=int(total_step),
    )

    obs, info = env.reset(seed=seed)
    return env, obs, info

In [2]:
import time
from trial_timing import reset_trial_times, append_trial_time

import os
import json
import numpy as np
from config import Config

def evenly_binary_sequence(count: int, n: int) -> np.ndarray:
\
\
       
    count = int(np.clip(count, 0, n))
    seq = np.zeros(n, dtype=np.int32)
    if count == 0:
        return seq
    if count == n:
        seq[:] = 1
        return seq

    acc = 0
    for i in range(n):
        acc += count
        if acc >= n:
            seq[i] = 1
            acc -= n
    return seq

def counts_to_binary_action_plan(
    counts_block: np.ndarray,
    num_od: int,
    total_step: int,
    block_steps: int
) -> np.ndarray:
\
\
\
       
    n_blocks = int(np.ceil(total_step / block_steps))
    action_plan = np.zeros((total_step, num_od), dtype=np.int32)

    for b in range(n_blocks):
        s = b * block_steps
        e = min((b + 1) * block_steps, total_step)
        n_local = e - s
        for od in range(num_od):
            c = int(np.rint(counts_block[b, od]))
            c = int(np.clip(c, 0, n_local))
            action_plan[s:e, od] = evenly_binary_sequence(c, n_local)

    return action_plan

def run_episode_with_plan(build_env_fn, worker_idx, action_plan, seed):
    env, obs, info = build_env_fn(
        worker_idx=worker_idx,
        total_step=action_plan.shape[0],
        seed=seed
    )
    total_reward = 0.0
    last_info = {}

    try:
        for t in range(action_plan.shape[0]):
            obs, reward, terminated, truncated, info = env.step(action_plan[t])
            total_reward += float(reward)
            last_info = info
            if terminated or truncated:
                break
    finally:
        env.close()

    trajectory = last_info.get("trajectory", [])
    return float(total_reward), trajectory

def de_optimize_counts_300s(
    build_env_fn,
    worker_idx,
    num_od,
    total_step,
    input_interval,
    detector_interval,
    init_points=0,
    n_iter=200,
    seed=42,
    verbose=2,
    popsize=None,
    F=0.8,
    CR=0.9,
    dither=(0.5, 0.9),
):
    block_steps = detector_interval // input_interval
    n_blocks = int(np.ceil(total_step / block_steps))

    var_meta = []
    lb = []
    ub = []
    for b in range(n_blocks):
        s = b * block_steps
        e = min((b + 1) * block_steps, total_step)
        n_local = e - s
        for od in range(num_od):
            var_meta.append((b, od, n_local))
            lb.append(0.0)
            ub.append(float(n_local))

    lb = np.array(lb, dtype=np.float32)
    ub = np.array(ub, dtype=np.float32)
    dim = lb.size

    rng = np.random.default_rng(seed)

    eval_budget = int(max(1, init_points + n_iter))

    def x_to_counts(x_vec: np.ndarray) -> np.ndarray:
        x_vec = np.asarray(x_vec, dtype=np.float32)
        counts = np.zeros((n_blocks, num_od), dtype=np.float32)
        for i, (b, od, n_local) in enumerate(var_meta):
            counts[b, od] = float(np.clip(x_vec[i], 0.0, float(n_local)))
        return counts

    eval_count = 0
    history = []

    def evaluate_x(x_vec: np.ndarray):
        nonlocal eval_count
        x_vec = np.clip(np.asarray(x_vec, dtype=np.float32), lb, ub)
        counts_block = x_to_counts(x_vec)
        action_plan = counts_to_binary_action_plan(
            counts_block=counts_block,
            num_od=num_od,
            total_step=total_step,
            block_steps=block_steps
        )
        score, _ = run_episode_with_plan(
            build_env_fn=build_env_fn,
            worker_idx=worker_idx,
            action_plan=action_plan,
            seed=seed + eval_count,
        )
        eval_count += 1
        return float(score)

    if popsize is None:
        popsize = int(np.clip(dim, 8, 40))
    popsize = int(max(4, min(popsize, eval_budget)))

    if eval_budget < 4:
        best_score = -np.inf
        best_x = None
        for _ in range(eval_budget):
            xr = rng.uniform(lb, ub).astype(np.float32)
            yr = evaluate_x(xr)
            history.append({"eval": eval_count, "type": "random_fallback", "score": yr})
            if yr > best_score:
                best_score = yr
                best_x = xr.copy()

        best_counts = x_to_counts(best_x)
        best_actions = counts_to_binary_action_plan(
            counts_block=best_counts,
            num_od=num_od,
            total_step=total_step,
            block_steps=block_steps
        )
        de_log = {
            "method": "DE",
            "eval_budget": eval_budget,
            "total_evals": eval_count,
            "dim": int(dim),
            "popsize": int(popsize),
            "history": history,
        }
        return best_actions, float(best_score), de_log, best_counts

    pop = rng.uniform(lb, ub, size=(popsize, dim)).astype(np.float32)
    scores = np.empty(popsize, dtype=np.float64)

    best_score = -np.inf
    best_x = None

    for i in range(popsize):
        scores[i] = evaluate_x(pop[i])
        history.append({"eval": eval_count, "phase": "init", "idx": i, "score": float(scores[i])})
        if scores[i] > best_score:
            best_score = float(scores[i])
            best_x = pop[i].copy()

    gen = 0

    while eval_count + popsize <= eval_budget:
        gen += 1
        new_pop = pop.copy()
        new_scores = scores.copy()

        Fg = rng.uniform(dither[0], dither[1]) if dither is not None else float(F)

        for i in range(popsize):

            idxs = np.arange(popsize)
            idxs = idxs[idxs != i]
            r1, r2, r3 = rng.choice(idxs, size=3, replace=False)

            mutant = pop[r1] + Fg * (pop[r2] - pop[r3])
            mutant = np.clip(mutant, lb, ub)

            cross_mask = rng.random(dim) < CR
            j_rand = rng.integers(0, dim)
            cross_mask[j_rand] = True

            trial = np.where(cross_mask, mutant, pop[i]).astype(np.float32)
            trial = np.clip(trial, lb, ub)

            trial_score = evaluate_x(trial)

            if trial_score >= scores[i]:
                new_pop[i] = trial
                new_scores[i] = trial_score

                if trial_score > best_score:
                    best_score = float(trial_score)
                    best_x = trial.copy()

        pop = new_pop
        scores = new_scores

        history.append({
            "gen": int(gen),
            "eval": int(eval_count),
            "best_score": float(best_score),
            "mean_score": float(np.mean(scores)),
            "F": float(Fg),
            "CR": float(CR),
        })

        if verbose >= 2 and (gen == 1 or gen % 5 == 0):
            print(f"[DE] gen {gen} | eval {eval_count}/{eval_budget} | best={best_score:.6f}")

    while eval_count < eval_budget:
        sigma = 0.10 * (ub - lb)
        xr = best_x + rng.normal(0.0, 1.0, size=dim).astype(np.float32) * sigma
        xr = np.clip(xr, lb, ub)
        yr = evaluate_x(xr)
        history.append({"eval": eval_count, "phase": "tail_local", "score": float(yr)})
        if yr > best_score:
            best_score = float(yr)
            best_x = xr.copy()

    best_counts = x_to_counts(best_x)
    best_actions = counts_to_binary_action_plan(
        counts_block=best_counts,
        num_od=num_od,
        total_step=total_step,
        block_steps=block_steps
    )

    de_log = {
        "method": "DE/rand/1/bin",
        "dim": int(dim),
        "eval_budget": int(eval_budget),
        "total_evals": int(eval_count),
        "popsize": int(popsize),
        "generations": int(gen),
        "F": float(F),
        "CR": float(CR),
        "dither": None if dither is None else [float(dither[0]), float(dither[1])],
        "history": history,
    }

    return best_actions, float(best_score), de_log, best_counts

os.makedirs(Config.RESULT_DIR, exist_ok=True)

_trial_result_dir = RESULT_DIR if "RESULT_DIR" in globals() else Config.RESULT_DIR
reset_trial_times(_trial_result_dir)

for trial in range(5):
    _trial_t0 = time.perf_counter()
    trial_idx = trial
    seed_opt = int(100 + (trial_idx + 1))
                                                                          
    seed_eval = trial

    best_actions, best_score, de_log, best_counts = de_optimize_counts_300s(
        build_env_fn=build_env,
        worker_idx=0,
        num_od=Config.NUM_OD,
        total_step=Config.TOTAL_STEP,
        input_interval=Config.INPUT_INTERVAL,
        detector_interval=Config.DETECTOR_INTERVAL,
        init_points=0,
        n_iter=600,
        seed=seed_opt,
        verbose=2,
        popsize=24,
        F=0.8,
        CR=0.9,
        dither=(0.5, 0.9),
    )

    print(f"[trial {trial}] Best total_reward:", best_score)
    print(f"[trial {trial}] Best action shape:", best_actions.shape)
    print(f"[trial {trial}] Best counts shape:", best_counts.shape)

    replay_reward, trajectory = run_episode_with_plan(
        build_env_fn=build_env,
        worker_idx=0,
        action_plan=best_actions,
        seed=seed_eval
    )
    print(f"[trial {trial}] Replay reward:", replay_reward)

    with open(os.path.join(Config.RESULT_DIR, f"trajectory_{trial}.json"), "w", encoding="utf-8") as f:
        json.dump(
            trajectory, f, ensure_ascii=False, indent=2,
            default=lambda o: o.tolist() if isinstance(o, np.ndarray) else o
        )

    with open(os.path.join(Config.RESULT_DIR, f"de_log_{trial}.json"), "w", encoding="utf-8") as f:
        json.dump(de_log, f, ensure_ascii=False, indent=2)
    append_trial_time(_trial_result_dir, trial, time.perf_counter() - _trial_t0)

ans: [11.  0.  3.  4.  5.  1. 20. 21.  4.], det: [23.  0. 17.  8. 12.  0. 37. 36.  7.], reward: -103.22222137451172
ans: [ 8. 11.  3. 14. 17.  9. 19. 30.  3.], det: [34. 10.  4. 28. 45. 36. 16. 55.  7.], reward: -337.4444580078125
ans: [23.  4. 13. 20. 17. 11. 15. 23.  5.], det: [20. 13. 17. 18. 48. 11. 66. 66. 12.], reward: -618.888916015625
ans: [15. 10. 10. 13. 17.  5. 11. 26.  2.], det: [20. 32.  7. 34. 35. 34. 63. 90.  9.], reward: -997.0
ans: [19.  4.  8. 21. 19. 14. 14. 33.  7.], det: [33. 20. 15. 31. 53. 23. 39. 65. 16.], reward: -396.4444580078125
ans: [14. 15. 15. 14. 12.  7. 13. 23.  6.], det: [53. 37. 28. 37. 40. 32. 53. 87. 10.], reward: -1091.5555419921875
Environment closed.
ans: [11.  0.  3.  4.  5.  1. 20. 21.  4.], det: [32.  0. 12.  2. 14.  0. 14. 20.  2.], reward: -72.11111450195312
ans: [ 8. 11.  3. 14. 17.  9. 19. 30.  3.], det: [25. 10. 12.  8. 25. 43. 30. 43.  4.], reward: -213.11111450195312
ans: [23.  4. 13. 20. 17. 11. 15. 23.  5.], det: [14.  9. 15. 19. 52. 

In [3]:
import os
import shutil
from config import Config


def remove_workers_root():
                                                            
    root = os.path.abspath(Config.WORKERS_ROOT)
    project_root = os.path.abspath(Config.RL_ROOT)
    if os.path.basename(root) != "workers":
        raise RuntimeError(f"Refusing to delete non-workers path: {root}")
    if os.path.commonpath([root, project_root]) != project_root:
        raise RuntimeError(f"Refusing to delete path outside experiment root: {root}")
    if os.path.isdir(root):
        shutil.rmtree(root)


remove_workers_root()
